# <center> <font color="#0036a3">Maestría en Inteligencia Artificial Aplicada (MNA)</font>  — New Version</center>

<center>

[![Materia](https://img.shields.io/badge/MATERIA-PROYECTO_INTEGRADOR-E0A800?style=for-the-badge&logoColor=white)](https://tec.mx)

</center>

<center>

[![Python](https://img.shields.io/badge/Python-3776AB?style=flat-square&logo=python&logoColor=white)](https://www.python.org/)
[![Jupyter](https://img.shields.io/badge/Jupyter-F37626?style=flat-square&logo=jupyter&logoColor=white)](https://jupyter.org/)
[![PyTorch](https://img.shields.io/badge/PyTorch-EE4C2C?style=flat-square&logo=pytorch&logoColor=white)](https://pytorch.org/)
[![OpenCV](https://img.shields.io/badge/OpenCV-5C3EE8?style=flat-square&logo=opencv&logoColor=white)](https://opencv.org/)
[![GitHub](https://img.shields.io/badge/Repo-GitHub-181717?style=flat-square&logo=github&logoColor=white)](https://github.com/jmtoral/proyecto_integrador_52)

</center>

## **<font color="#0036a3">Avance 4 — Evaluación de Image Enhancement (Retinex, EndoLMSPEC e IAT/EndoViT) sobre Múltiples Modelos de Profundidad Endoscópica en SCARED</font>**

### **<font color="#E0A800">Proyecto Integrador — TC5035.10</font>**

---

## **<center> <font color="#0036a3">Equipo 52</font> </center>**

<table style="border-collapse:collapse; width:60%; margin:auto;">
  <tr>
    <td align="center" style="border:none; width:33%; padding:10px;">
      <img src="https://raw.githubusercontent.com/jmtoral/proyecto_integrador_52/main/reports/elda.jpg" width="80" height="80" style="border-radius:50%; object-fit:cover; object-position:center top;"><br>
      <strong>Elda Morales</strong><br><small>A00449074</small>
    </td>
    <td align="center" style="border:none; width:33%; padding:10px;">
      <img src="https://raw.githubusercontent.com/jmtoral/proyecto_integrador_52/main/reports/mpgc.jpg" width="80" height="80" style="border-radius:50%; object-fit:cover; object-position:center top;"><br>
      <strong>María Paula Gutiérrez</strong><br><small>A01747706</small>
    </td>
    <td align="center" style="border:none; width:33%; padding:10px;">
      <img src="https://raw.githubusercontent.com/jmtoral/proyecto_integrador_52/main/reports/jmtc_n.jpg" width="80" height="80" style="border-radius:50%; object-fit:cover; object-position:center top;"><br>
      <strong>José Manuel Toral</strong><br><small>A01122243</small>
    </td>
  </tr>
</table>

---

### Objetivos de este notebook

Evaluar si los modelos de **Image Enhancement** mejoran el desempeño de **múltiples modelos de depth estimation** sobre SCARED:

1. Comparar **4 métodos de enhancement**: None (baseline), Retinex SSR, EndoLMSPEC e IAT/EndoViT
2. Evaluar sobre **2 modelos de depth**: Endo-Depth (Recasens et al., 2021) y EndoSfMLearner (Ozyoruk et al., 2020)
3. Cuantificar el impacto en: **AbsRel, RMSE, Chamfer Distance**
4. Analizar si la mejora se concentra en **zonas especulares** (hipótesis central)
5. Evaluar **viabilidad clínica** (FPS)

> García-Vega, A., et al. (2022). Multi-Scale Structural-aware Exposure Correction for Endoscopic Imaging. *arXiv:2210.15033*.
> Rahman, Z., et al. (2004). Retinex processing for automatic image enhancement. *Journal of Electronic Imaging*, 13(1). https://doi.org/10.1117/1.1636183
> Wang, T., et al. (2022). Ultra-High-Definition Low-Light Image Enhancement. *AAAI 2022*.
> Ozyoruk, K.B., et al. (2020). EndoSLAM Dataset and Endo-SfMLearner. *arXiv:2006.16670*.

## New Version — Evaluación con el *split* oficial y Endo-STTN

> **Equipo 52 · Proyecto Integrador TC5035.10 — MNA, Tecnológico de Monterrey**

Esta **New Version** del Avance 4 incorpora las dos peticiones del profesor que modifican la metodología de evaluación:

1. **Protocolo de prueba estándar.** Evaluamos sobre el ***split* oficial de AF-SfMLearner** (`splits/endovis/test_files.txt`): **550 fotogramas** de los *datasets* 1–7 de SCARED, el mismo conjunto que reporta la literatura → resultados **directamente comparables**.

2. **Endo-STTN como nuevo método de mejora.** Quinto *enhancement*: **Endo-STTN** (*Spatio-Temporal Transformer Network*), que elimina **reflejos especulares** usando información **temporal**.

> **Diferencia con el Avance 5:** este Avance 4 usa los **pesos originales** de cada artículo (Hamlyn, KITTI, Endovis), *no* re-entrenados en SCARED. La comparación A4 (pesos originales) vs. A5 (re-entrenados en SCARED) es uno de los aportes centrales del proyecto.

### Pregunta de investigación

> *¿Los métodos de realce de imagen mejoran la estimación de profundidad monocular en endoscopía?*

### Flujo general

```mermaid
flowchart LR
    A[SCARED zip<br/>rgb.mp4 + scene_points] --> B[Extracción<br/>550 frames del split]
    B --> C{Enhancement}
    C -->|none / retinex / endolmspec<br/>iat / endosttn| D[Modelo de<br/>profundidad]
    D --> E[Profundidad predicha]
    E --> F[Métricas vs GT]
    F --> G[Tabla + CSV + viz]
```

---
## 0. Pipeline general del experimento

```mermaid
flowchart TD
    A["SCARED - datasets 8-9, keyframes 0-4"] --> B
    B["Imagen RGB 1280x1024 px"] --> C1 & C2 & C3 & C4

    subgraph ENHANCEMENT ["Image Enhancement"]
        C1["None - baseline"]
        C2["Retinex SSR - Rahman 2004"]
        C3["EndoLMSPEC - Endo4IE - Garcia-Vega 2022"]
        C4["IAT EndoViT - Endo4IE - Wang 2022"]
    end

    C1 & C2 & C3 & C4 --> D1 & D2 & D3 & D4 & D5

    subgraph DEPTH ["Modelos de Depth"]
        D1["Endo-Depth - ResNet18 - Recasens 2021"]
        D2["EndoSfMLearner - DispResNet18 - Ozyoruk 2020"]
        D3["AF-SfMLearner - ResNet18 + AppFlow - Shao 2022"]
        D4["HADepth - DINOv2 ViT-Base + DoRA - Shao en revision"]
        D5["MonoViT - MPViT-Small - Zhao 2022"]
    end

    D1 & D2 & D3 & D4 & D5 --> E["Median Scaling a mm"]
    E --> F["AbsRel - RMSE - Chamfer - FPS"]

    style ENHANCEMENT fill:#fff8e1,stroke:#E0A800,stroke-width:2px
    style DEPTH fill:#e8f4fd,stroke:#2766CB,stroke-width:2px
```

**Diseño factorial**: 4 enhancements × 5 modelos de depth × 10 keyframes = **200 evaluaciones**

### Hipótesis
Los métodos de enhancement reducen el error de profundidad con mayor efecto en zonas especulares. HADepth y AF-SfMLearner, diseñados explícitamente para endoscopía con highlight-awareness y appearance flow, deberían ser los más robustos al enhancement por diseño.

In [1]:
import subprocess, sys

# Solo lo que Colab no trae por defecto
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "tifffile", "scikit-image"])

import torch, tifffile, cv2, numpy as np
print(f"torch    : {torch.__version__}")
print(f"tifffile : {tifffile.__version__}")
print(f"opencv   : {cv2.__version__}")
print(f"CUDA OK  : {torch.cuda.is_available()}")

torch    : 2.11.0+cu128
tifffile : 2026.4.11
opencv   : 4.13.0
CUDA OK  : True


---
## 1. Configuración de rutas

### Estructura esperada en Google Drive

```
MyDrive/proyecto_integrador/
├── scared_raw/
├── endo_depth_weights/            ← encoder.pth + depth.pth  (Endo-Depth)
├── Endo-Depth-and-Motion/
├── Model_MIA/                     ← encoder.pth + depth.pth  (AF-SfMLearner Stage-wise)
├── HADepth/                       ← repo completo (código: models/, utils/, ...)
├── HADepth_fullmodel/             ← depth_model.pth
├── EndoLMSPEC/
│   └── checkpoint/main_net/model_256_combined_SSIM5_1.pth
├── EndoViT/
│   └── Endo4IE/best_Epoch50_laplacian_histogan_loss.pth
├── EndoSLAM/
│   ├── EndoSfMLearner/
│   └── pretrained/08-13-00_00/dispnet_model_best.pth.tar
├── MonoViT/
│   ├── networks/                  ← mpvit.py (parcheado), hr_decoder.py
│   └── mono_640x192/encoder.pth + depth.pth
└── avance4_outputs/
```

### Modelos de depth — resumen de pesos

| Modelo | Archivo(s) de pesos | Backbone | Dataset entrenamiento | AbsRel paper† |
|---|---|---|---|---|
| **Endo-Depth** | `endo_depth_weights/encoder.pth` + `depth.pth` | ResNet-18 | Hamlyn (laparoscopy) | 0.094* |
| **EndoSfMLearner** | `EndoSLAM/pretrained/.../dispnet_model_best.pth.tar` | ResNet-18 (DispResNet) | EndoSLAM (capsule) | 0.062 |
| **AF-SfMLearner** | `Model_MIA/encoder.pth` + `depth.pth` | ResNet-18 + Appearance Flow | Endovis | 0.059 |
| **HADepth** | `HADepth_fullmodel/depth_model.pth` | DINOv2 ViT-Base + DoRA | Endovis | 0.049 |
| **MonoViT** | `MonoViT/mono_640x192/encoder.pth` + `depth.pth` | MPViT-Small (ViT) | KITTI (outdoor) | — |

† Números **reportados en los papers originales** en sus propios splits de evaluación, sin enhancement. No son resultados de este experimento. Nuestros resultados se miden sobre datasets 8–9 de SCARED con protocolo unificado.  
\* EndoDepth-Perf según tabla de HADepth; Endo-Depth original evaluado en Hamlyn.

### Métodos de enhancement — resumen de pesos

| Método | Archivo de pesos | Arquitectura | Dataset entrenamiento | Parámetros |
|---|---|---|---|---|
| **Retinex SSR** | — (sin pesos) | Filtro gaussiano logarítmico | — | 0 |
| **EndoLMSPEC** | `EndoLMSPEC/checkpoint/main_net/model_256_combined_SSIM5_1.pth` | U-Net multi-escala + pirámide Laplaciana | Endo4IE (over+under) | ~2 M |
| **IAT / EndoViT** | `EndoViT/Endo4IE/best_Epoch50_laplacian_histogan_loss.pth` | Transformer local+global (IAT) | Endo4IE fine-tune, época 50 | ~90 K |

## 1. Configuración del entorno y el *split* oficial

Define las rutas (Colab→Drive, local→`E:`) y carga el ***split* oficial**. Cada línea de `test_files.txt` tiene el formato:

```
dataset3/keyframe4 390 l   →   (dataset_3, keyframe_4, frame 390, cámara izquierda)
```

`load_split()` traduce esas 550 líneas a tuplas que consume el resto del *notebook*. Usar este *split* es lo que hace que los números sean **comparables con el estado del arte**.

In [ ]:
from pathlib import Path
import sys, subprocess

IN_COLAB = "google.colab" in sys.modules
if not IN_COLAB:
    try:
        import google.colab
        IN_COLAB = True
    except ImportError:
        IN_COLAB = False

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    BASE          = Path("/content/drive/MyDrive/proyecto_integrador")
    MODEL_PATH    = BASE / "endo_depth_weights"
    SCARED_ROOT   = BASE / "scared_raw"
    EDAM_PATH     = BASE / "Endo-Depth-and-Motion"
    LMSPEC_PATH   = BASE / "EndoLMSPEC"
    IAT_PATH      = BASE / "EndoViT"
    ENDOSLAM_PATH = BASE / "EndoSLAM"
    MONOVIT_PATH  = BASE / "MonoViT"
    AFSFM_WEIGHTS = BASE / "Model_MIA"
    HADEPTH_PATH  = BASE / "HADepth"
    HADEPTH_WEIGHTS = BASE / "HADepth_fullmodel"
    STTN_PATH     = BASE / "Endo-STTN"
    REPO_ROOT = Path("/content/repo_52")
    if not REPO_ROOT.exists():
        subprocess.check_call([
            "git", "clone", "--depth=1",
            "https://github.com/jmtoral/proyecto_integrador_52.git",
            str(REPO_ROOT)
        ])
    else:
        # repo ya clonado en una sesion previa: actualizar para traer data/splits, etc.
        subprocess.check_call(["git", "-C", str(REPO_ROOT), "pull", "--ff-only"])
    subprocess.check_call(["git","-C",str(REPO_ROOT),"config","user.email","jmtoralcruz@gmail.com"])
    subprocess.check_call(["git","-C",str(REPO_ROOT),"config","user.name","jmtoral"])
    OUT_DIR    = REPO_ROOT / "outcomes" / "avance4_newversion"
    SPLIT_FILE = REPO_ROOT / "data" / "splits" / "endovis" / "test_files.txt"
    FRAMES_CACHE = Path("/content/split_frames")
else:
    MODEL_PATH      = Path("E:/endo_depth_weights")
    SCARED_ROOT     = Path("D:/Proyecto_Integrador/Corrreccion_Luz/data/scared_raw")
    EDAM_PATH       = Path("E:/Endo-Depth-and-Motion")
    LMSPEC_PATH     = Path("E:/EndoLMSPEC")
    IAT_PATH        = Path("E:/EndoVit")
    ENDOSLAM_PATH   = Path("E:/EndoSLAM")
    MONOVIT_PATH    = Path("E:/MonoViT")
    AFSFM_WEIGHTS   = Path("E:/Model_MIA")
    HADEPTH_PATH    = Path("E:/HADepth")
    HADEPTH_WEIGHTS = Path("E:/HADepth_fullmodel")
    STTN_PATH       = Path("E:/Endo-STTN")
    REPO_ROOT       = Path(r"d:\Proyecto_Integrador\Corrreccion_Luz")
    OUT_DIR         = REPO_ROOT / "outcomes" / "avance4_newversion"
    SPLIT_FILE      = REPO_ROOT / "data" / "splits" / "endovis" / "test_files.txt"
    FRAMES_CACHE    = REPO_ROOT / "data" / "split_frames"

LMSPEC_WEIGHTS  = LMSPEC_PATH / "checkpoint" / "main_net" / "model_256_combined_SSIM5_1.pth"
IAT_WEIGHTS     = IAT_PATH / "Endo4IE" / "best_Epoch50_laplacian_histogan_loss.pth"
ENDOSFM_WEIGHTS = ENDOSLAM_PATH / "pretrained" / "08-13-00_00" / "dispnet_model_best.pth.tar"
MONOVIT_WEIGHTS = MONOVIT_PATH / "mono_640x192"

# Endo-STTN
STTN_CKPT_DIR    = STTN_PATH / "release_model" / "pretrained_model"
STTN_CKPT_NUMBER = "9"
STTN_WEIGHTS     = STTN_CKPT_DIR / "gen_00009.pth"
STTN_GDRIVE_ID   = "14sdaDejsxgRuzHBSuqH2xEpbxuqyWI-R"

OUT_DIR.mkdir(parents=True, exist_ok=True)
FRAMES_CACHE.mkdir(parents=True, exist_ok=True)
CAP_MM = 150.0

def load_split(split_file):
    items = []
    with open(split_file) as f:
        for line in f:
            line = line.strip()
            if not line: continue
            folder, frame_id, _side = line.split()
            ds, kf = folder.split("/")
            ds_n = "dataset_" + ds.replace("dataset","")
            kf_n = "keyframe_" + kf.replace("keyframe","")
            items.append((ds_n, kf_n, int(frame_id)))
    return items

SPLIT_ITEMS = load_split(SPLIT_FILE)

print(f"Entorno : {'Google Colab' if IN_COLAB else 'Local'}")
print(f"OUT_DIR : {OUT_DIR}")
print(f"Split   : {len(SPLIT_ITEMS)} frames  (datasets {sorted({d for d,_,_ in SPLIT_ITEMS})})")
for name, p in [("MODEL_PATH", MODEL_PATH), ("SCARED_ROOT", SCARED_ROOT),
                ("LMSPEC_WEIGHTS", LMSPEC_WEIGHTS), ("IAT_WEIGHTS", IAT_WEIGHTS),
                ("ENDOSFM_WEIGHTS", ENDOSFM_WEIGHTS), ("MONOVIT_WEIGHTS", MONOVIT_WEIGHTS),
                ("AFSFM_WEIGHTS", AFSFM_WEIGHTS), ("HADEPTH_PATH", HADEPTH_PATH),
                ("HADEPTH_WEIGHTS", HADEPTH_WEIGHTS), ("STTN_PATH", STTN_PATH)]:
    print(f"  {name:18s}: {'OK' if p.exists() else 'NO ENCONTRADO'}")


---
### Extracción del split oficial (550 frames)

El split de AF-SfMLearner (`endovis/test_files.txt`) referencia **frames concretos dentro de los vídeos** de cada *keyframe* (`data/rgb.mp4`) y su *ground-truth* (`data/scene_points.tar.gz`). Esta celda extrae solo los frames del split —imagen + GT recortados a la mitad izquierda `[0:1024,:]`— y los cachea.

Mapeo oficial: frame `N` → imagen = frame `N` del vídeo, GT = `scene_points{N-1:06d}.tiff`.

## 2. Extracción de los 550 fotogramas del *split*

Los fotogramas del *split* **no están sueltos** en los `.zip` de SCARED: viven dentro de `data/rgb.mp4` (vídeo) y `data/scene_points.tar.gz` (*ground-truth*, un `.tiff` por fotograma). Esta celda los extrae siguiendo la convención oficial de AF-SfMLearner:

- **Imagen** del fotograma `N` → fotograma `N` del `rgb.mp4`
- **Profundidad** del fotograma `N` → `scene_points{N-1:06d}.tiff` (índice **desfasado en 1**)
- Recorte a la **mitad izquierda** `[0:1024, :]`; del `.tiff` se toma el **canal Z** (mm)

**Caché persistente:** la primera vez extrae y guarda todo en `BASE/split_frames.npz` (Drive); en sesiones siguientes **carga en segundos** sin re-extraer (`FORCE_REEXTRACT = True` para forzar).

In [ ]:
import io, zipfile, tarfile, cv2, numpy as np, tifffile
from collections import defaultdict
from tqdm.notebook import tqdm

# Caché persistente empaquetado en Drive (sobrevive entre sesiones de Colab)
NPZ_CACHE = BASE / "split_frames.npz"
def _key(ds, kf, fid): return f"{ds}|{kf}|{fid}"

def _frame_paths(ds, kf, fid):
    base = FRAMES_CACHE / ds / kf
    return base / f"img_{fid:010d}.png", base / f"gt_{fid:010d}.npy"

def extract_split_frames(split_items, scared_root, force=False):
    """Extrae imagen+GT de cada (ds,kf,frame). Scratch en FRAMES_CACHE (rapido)."""
    by_kf = defaultdict(list)
    for ds, kf, fid in split_items:
        by_kf[(ds, kf)].append(fid)
    n_done = 0
    for (ds, kf), fids in tqdm(by_kf.items(), desc="Keyframes del split"):
        fids = sorted(set(fids))
        pend = [f for f in fids if force or not _frame_paths(ds, kf, f)[0].exists()
                                        or not _frame_paths(ds, kf, f)[1].exists()]
        if not pend: continue
        (FRAMES_CACHE / ds / kf).mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(scared_root / f"{ds}.zip") as z:
            sp_bytes = z.read(f"{ds}/{kf}/data/scene_points.tar.gz")
            with tarfile.open(fileobj=io.BytesIO(sp_bytes)) as t:
                tnames = {n.split("/")[-1]: n for n in t.getnames() if n.endswith(".tiff")}
                gt_cache = {}
                for fid in pend:
                    key = f"scene_points{fid-1:06d}.tiff"
                    if key not in tnames: gt_cache[fid] = None; continue
                    raw = t.extractfile(tnames[key]).read()
                    tiff = tifffile.imread(io.BytesIO(raw))
                    dz = tiff[..., 2].astype(np.float32) if tiff.ndim == 3 else tiff.astype(np.float32)
                    dz = dz[0:1024, :]; dz[dz <= 0] = np.nan; gt_cache[fid] = dz
            tmp_mp4 = FRAMES_CACHE / "_tmp.mp4"
            with open(tmp_mp4, "wb") as fout: fout.write(z.read(f"{ds}/{kf}/data/rgb.mp4"))
            cap = cv2.VideoCapture(str(tmp_mp4)); want = set(pend); maxf = max(pend)
            idx = 0; got = {}
            while idx <= maxf:
                ok, frame = cap.read()
                if not ok: break
                if idx in want: got[idx] = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)[0:1024, :]
                idx += 1
            cap.release(); tmp_mp4.unlink(missing_ok=True)
        for fid in pend:
            img_p, gt_p = _frame_paths(ds, kf, fid)
            if fid in got: cv2.imwrite(str(img_p), cv2.cvtColor(got[fid], cv2.COLOR_RGB2BGR))
            if gt_cache.get(fid) is not None: np.save(gt_p, gt_cache[fid])
            n_done += 1
    print(f"Extracción lista — {n_done} frames nuevos")

# Diccionario en memoria: clave -> (img_uint8, gt_float32|None)
SPLIT_DATA = {}

if NPZ_CACHE.exists() and not globals().get("FORCE_REEXTRACT", False):
    # ---- Carga rapida desde Drive (segundos) ----
    print(f"Cargando caché empaquetado: {NPZ_CACHE.name}")
    _npz = np.load(NPZ_CACHE, allow_pickle=True)
    for ds, kf, fid in SPLIT_ITEMS:
        k = _key(ds, kf, fid)
        ik, gk = "img_"+k, "gt_"+k
        if ik in _npz.files:
            gt = _npz[gk] if gk in _npz.files else None
            if gt is not None and gt.size == 1 and np.isnan(gt).all(): gt = None
            SPLIT_DATA[k] = (_npz[ik], gt)
    print(f"Caché cargado — {len(SPLIT_DATA)} frames listos (sin re-extraer)")
else:
    # ---- Primera vez: extraer y empaquetar en Drive ----
    extract_split_frames(SPLIT_ITEMS, SCARED_ROOT)
    save_dict = {}
    for ds, kf, fid in SPLIT_ITEMS:
        img_p, gt_p = _frame_paths(ds, kf, fid)
        if not img_p.exists(): continue
        k = _key(ds, kf, fid)
        img = cv2.cvtColor(cv2.imread(str(img_p), cv2.IMREAD_COLOR), cv2.COLOR_BGR2RGB)
        gt = np.load(gt_p) if gt_p.exists() else np.array([np.nan], np.float32)
        save_dict["img_"+k] = img
        save_dict["gt_"+k] = gt
        SPLIT_DATA[k] = (img, gt if gt.size > 1 else None)
    np.savez_compressed(NPZ_CACHE, **save_dict)
    print(f"Caché empaquetado guardado en Drive: {NPZ_CACHE}  ({len(SPLIT_DATA)} frames)")
    print("En sesiones futuras se cargara de aqui en segundos (no re-extrae).")


---
## 2. Cargar Endo-Depth

Mismo modelo que en Avances 3 y 4: ResNet18 encoder + DepthDecoder, pesos Hamlyn. No se modifica el modelo — es la variable dependiente fija del experimento.

In [3]:
import torch
import numpy as np

sys.path.insert(0, str(EDAM_PATH / "apps" / "depth_estimate"))
from resnet_encoder import ResnetEncoder
from depth_decoder import DepthDecoder

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Dispositivo: {DEVICE}")

encoder = ResnetEncoder(18, False)
loaded_enc = torch.load(MODEL_PATH / "encoder.pth", map_location=DEVICE)
FEED_HEIGHT = loaded_enc["height"]
FEED_WIDTH  = loaded_enc["width"]
filtered_enc = {k: v for k, v in loaded_enc.items() if k in encoder.state_dict()}
encoder.load_state_dict(filtered_enc)
encoder.to(DEVICE).eval()

depth_decoder = DepthDecoder(num_ch_enc=encoder.num_ch_enc, scales=range(4))
loaded_dec = torch.load(MODEL_PATH / "depth.pth", map_location=DEVICE)
depth_decoder.load_state_dict(loaded_dec)
depth_decoder.to(DEVICE).eval()

print(f"Resolución del modelo: {FEED_HEIGHT}×{FEED_WIDTH}")
print("Endo-Depth cargado ✓")

Dispositivo: cuda


/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:135: UserWarning: Using 'weights' as positional parameter(s) is deprecated since 0.13 and may be removed in the future. Please use keyword parameter(s) instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


Resolución del modelo: 256×320
Endo-Depth cargado ✓


In [4]:
import sys, torch, torch.nn.functional as F, numpy as np
import importlib.util

# EndoSLAM tiene models/__init__.py con imports relativos (from .DispResNet import ...).
# Para que funcionen, el modulo debe estar en sys.modules ANTES de exec_module.
_endosfm_dir = ENDOSLAM_PATH / 'EndoSfMLearner'
if str(_endosfm_dir) not in sys.path:
    sys.path.insert(0, str(_endosfm_dir))

_spec = importlib.util.spec_from_file_location(
    '_endosfm_models',
    str(_endosfm_dir / 'models' / '__init__.py'),
    submodule_search_locations=[str(_endosfm_dir / 'models')]
)
endosfm_models = importlib.util.module_from_spec(_spec)
sys.modules['_endosfm_models'] = endosfm_models   # registrar ANTES de exec
_spec.loader.exec_module(endosfm_models)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

endosfm_net = endosfm_models.DispResNet(18, False).to(DEVICE)
weights = torch.load(ENDOSFM_WEIGHTS, map_location=DEVICE)
endosfm_net.load_state_dict(weights['state_dict'])
endosfm_net.eval()

n_params = sum(p.numel() for p in endosfm_net.parameters())
print(f'EndoSfMLearner parametros: {n_params/1e6:.2f} M')
print(f'Pesos: {ENDOSFM_WEIGHTS.name}')
print('EndoSfMLearner cargado OK')


EndoSfMLearner parámetros: 14.84 M
Pesos: dispnet_model_best.pth.tar
EndoSfMLearner cargado ✓


---
## 2b. EndoSfMLearner — Endo-SfMLearner (Ozyoruk et al., 2020)

**EndoSfMLearner** es un método de estimación de profundidad monocular auto-supervisado específicamente diseñado para endoscopía, publicado como parte del dataset EndoSLAM. Sus contribuciones principales son:

- **Brightness-aware photometric loss**: hace la predicción de profundidad robusta a variaciones de iluminación — relevante directamente para nuestra hipótesis de corrección de iluminación
- **Spatial attention-based pose network**: optimizado para las características geométricas de imágenes endoscópicas

La arquitectura usa **DispResNet** (ResNet-18 como backbone), misma familia que Endo-Depth, pero con una cabeza de disparidad diferente y entrenado en datos de endoscopía de cápsula (EndoSLAM dataset).

> Ozyoruk, K. B., et al. (2020). EndoSLAM Dataset and An Unsupervised Monocular Visual Odometry and Depth Estimation Approach for Endoscopic Videos: Endo-SfMLearner. *arXiv:2006.16670*.

In [ ]:
import torch

# AF-SfMLearner usa ResnetEncoder + DepthDecoder de Monodepth2 — misma arquitectura
# que Endo-Depth. Reutilizamos las clases ya importadas; solo cargamos otros pesos.
afsfm_encoder = ResnetEncoder(18, False)
loaded_enc = torch.load(AFSFM_WEIGHTS / "encoder.pth", map_location=DEVICE)
AFSFM_H = loaded_enc.get("height", 256)
AFSFM_W = loaded_enc.get("width",  320)
filtered_enc = {k: v for k, v in loaded_enc.items() if k in afsfm_encoder.state_dict()}
afsfm_encoder.load_state_dict(filtered_enc)
afsfm_encoder.to(DEVICE).eval()

afsfm_decoder = DepthDecoder(num_ch_enc=afsfm_encoder.num_ch_enc, scales=range(4))
loaded_dec = torch.load(AFSFM_WEIGHTS / "depth.pth", map_location=DEVICE)
afsfm_decoder.load_state_dict(loaded_dec)
afsfm_decoder.to(DEVICE).eval()

n_params = sum(p.numel() for p in afsfm_encoder.parameters()) + \
           sum(p.numel() for p in afsfm_decoder.parameters())
print(f"AF-SfMLearner parámetros: {n_params/1e6:.2f} M")
print(f"Resolución              : {AFSFM_H}×{AFSFM_W}")
print("AF-SfMLearner cargado ✓")

---
## 2c. AF-SfMLearner — Appearance Flow para Endoscopía (Shao et al., 2022)

**AF-SfMLearner** (*Self-Supervised Monocular Depth and Ego-Motion Estimation in Endoscopy: Appearance Flow to the Rescue*, Medical Image Analysis 2022) introduce un módulo de **appearance flow** que compensa explícitamente los reflejos especulares y las variaciones de iluminación endoscópica durante el entrenamiento auto-supervisado.

La contribución clave frente a EndoSfMLearner y Endo-Depth:

- **Appearance flow module**: predice un campo de flujo denso que mapea píxeles especulares a regiones de tejido cercanas con iluminación similar, generando una imagen de referencia "limpia" para la pérdida fotométrica
- **Transformer de apariencia**: deforma explícitamente la imagen fuente antes de comparar con el frame objetivo — hace el entrenamiento robusto a brillos sin depender de datos con anotación manual

La arquitectura del encoder y decoder es la misma que Endo-Depth (ResNet-18 + DepthDecoder de Monodepth2), lo que permite comparación directa; la diferencia está en la **función de pérdida y el módulo auxiliar de apariencia** que solo actúa durante el entrenamiento.

| Checkpoint | AbsRel | Sq Rel | RMSE | Dataset evaluación |
|---|---|---|---|---|
| Stage-wise (**este experimento**) | 0.059 | 0.435 | 4.925 | Endovis |
| End-to-end | 0.059 | 0.470 | 5.062 | Endovis |
| ICRA 2021 | 0.063 | 0.489 | 5.185 | Endovis |

> Shao, S., Pei, Z., Chen, W., Zhu, W., Wu, X., Sun, D., & Zhang, B. (2022). Self-Supervised Monocular Depth and Ego-Motion Estimation in Endoscopy: Appearance Flow to the Rescue. *Medical Image Analysis*, 77, 102338.

In [ ]:
import subprocess, torch, types

# Dependencia de HADepth (DINOv2 backbone)
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'fvcore'])

# MAB.py esta ausente del repo pero from .MAB import * no aporta nada
# a ResBottleneckBlock (que solo usa torch y LayerNorm del mismo archivo).
# Creamos un stub vacio para que el import no falle.
_mab_path = HADEPTH_PATH / 'models' / 'backbones' / 'layers' / 'MAB.py'
if not _mab_path.exists():
    _mab_path.write_text('# stub: MAB no se usa en inferencia\n')
    print('MAB.py stub creado')

# --- 1. Limpiar models.* para partir de cero ---
for _k in list(sys.modules.keys()):
    if _k == 'models' or _k.startswith('models.') \
    or _k == 'utils'  or _k.startswith('utils.'):
        del sys.modules[_k]

# --- 2. Inyectar namespace package 'models' apuntando a HADepth/models/ ---
_models_ns = types.ModuleType('models')
_models_ns.__path__ = [str(HADEPTH_PATH / 'models')]
_models_ns.__package__ = 'models'
sys.modules['models'] = _models_ns

_hadepth_str = str(HADEPTH_PATH)
sys.path = [p for p in sys.path if p != _hadepth_str]
sys.path.insert(0, _hadepth_str)

# --- 3. Importar y cargar HADepth ---
import models.hadepth as hadepth_module

hadepth_net = hadepth_module.hadepth(
    backbone_size='base',
    r=4,
    lora_type='dora',
    image_shape=(224, 280),
    pretrained_path=None,
    residual_block_indexes=[2,5,8,11],
    include_cls_token=True,
)

depther_dict = torch.load(HADEPTH_WEIGHTS / 'depth_model.pth', map_location=DEVICE)
model_dict = hadepth_net.state_dict()
hadepth_net.load_state_dict({k: v for k, v in depther_dict.items() if k in model_dict})
hadepth_net.to(DEVICE).eval()

# --- 4. Limpiar sys.modules para no bloquear EndoSfMLearner ---
for _k in list(sys.modules.keys()):
    if _k == 'models' or _k.startswith('models.') \
    or _k == 'utils'  or _k.startswith('utils.'):
        del sys.modules[_k]

n_params = sum(p.numel() for p in hadepth_net.parameters())
print(f'HADepth parametros: {n_params/1e6:.1f} M')
print('Backbone          : DINOv2 ViT-Base + DoRA (rank=4) + DPTHead')
print('Imagen interna    : 224x280 px (resize automatico en forward)')
print('HADepth cargado OK')


---
## 2d. HADepth — Highlight-Aware Monocular Depth para Endoscopía (Shao et al., en revisión)

**HADepth** (*Highlight-aware monocular depth estimation for endoscopy*, Signal, Image and Video Processing) diseña un pipeline explícitamente consciente de los **reflejos especulares** (highlights), el principal factor de degradación en imágenes endoscópicas.

### Arquitectura

```mermaid
flowchart LR
    IN["Imagen RGB\n224×280 px\n(resize interno)"] --> VIT

    subgraph ENCODER ["Encoder: DINOv2 ViT-Base + DoRA"]
        VIT["ViT-Base/14\n12 bloques Transformer\npatch 14×14"]
        DORA["DoRA rank=4\nen MLP de cada bloque\n(fine-tune eficiente)"]
        VIT --- DORA
    end

    VIT -->|"features de capas 2,5,8,11"| DPT

    subgraph DECODER ["Decoder: DPTHead"]
        DPT["4 proyecciones\n+ resize/deconv"]
        RF["4 RefineNet\nFusion Blocks"]
        DPT --> RF
    end

    RF --> OUT["outputs disp 0..3\nSigmoid"]

    style ENCODER fill:#e8f4fd,stroke:#2766CB
    style DECODER fill:#fff8e1,stroke:#E0A800
```

**Contribuciones clave:**
- **DINOv2** como backbone: representaciones de largo rango más robustas a discontinuidades de iluminación que ResNet
- **DoRA** (*Weight-Decomposed Low-Rank Adaptation*): fine-tune eficiente del ViT en datos endoscópicos
- **DPTHead** multi-escala: fusión de features de 4 capas del transformer

### Referencia de resultados (paper, sin enhancement)

> Los números siguientes son **reportados por los autores** en su split de Endovis — **no son resultados de nuestro experimento**. Los usamos solo como referencia de contexto para calibrar expectativas.

| Método | AbsRel | Sq Rel | RMSE | RMSE log |
|---|---|---|---|---|
| Monodepth2 | 0.071 | 0.590 | 5.606 | 0.094 |
| EndoSfMLearner | 0.062 | 0.606 | 5.726 | 0.093 |
| AF-SfMLearner | 0.059 | 0.435 | 4.925 | 0.082 |
| HADepth (paper) | **0.049** | **0.326** | **4.286** | **0.069** |

Nuestros resultados se medirán sobre **datasets 8–9 de SCARED con y sin enhancement** usando `compute_metrics` con median scaling, protocolo idéntico para todos los modelos del experimento.

> Shao, S., et al. HADepth: Highlight-aware monocular depth estimation for endoscopy. *Signal, Image and Video Processing* (en revisión).

In [5]:
import sys
import torch
import numpy as np

# timm y einops son necesarios para MPViT
import subprocess
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "timm", "einops"])

# Agregar MonoViT al path — mpvit.py ya fue parcheado (sin mmcv/mmseg)
sys.path.insert(0, str(MONOVIT_PATH))
from networks.nets import DeepNet

# DeepNet encapsula encoder + decoder juntos (no separarlos)
monovit_net = DeepNet(type='mpvitnet')

# Cargar pesos del encoder (contiene height/width de entrenamiento)
enc_weights = torch.load(MONOVIT_WEIGHTS / "encoder.pth", map_location=DEVICE)
MONOVIT_H = enc_weights.get("height", 192)
MONOVIT_W = enc_weights.get("width",  640)

# Cargar encoder en monovit_net.encoder
filtered_enc = {k.replace("encoder.", ""): v
                for k, v in enc_weights.items()
                if k.replace("encoder.", "") in monovit_net.encoder.state_dict()}
monovit_net.encoder.load_state_dict(filtered_enc, strict=False)

# Cargar decoder en monovit_net.decoder
dec_weights = torch.load(MONOVIT_WEIGHTS / "depth.pth", map_location=DEVICE)
monovit_net.decoder.load_state_dict(dec_weights)

monovit_net.to(DEVICE).eval()

# Aliases para compatibilidad con predict_depth_monovit
monovit_enc = monovit_net.encoder
monovit_dec = monovit_net.decoder

n_params = sum(p.numel() for p in monovit_net.parameters())
print(f"MonoViT parámetros: {n_params/1e6:.1f} M")
print(f"Resolución        : {MONOVIT_H}×{MONOVIT_W}")
print("MonoViT cargado ✓")

/usr/local/lib/python3.12/dist-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)


MonoViT parámetros: 27.9 M
Resolución        : 192×640
MonoViT cargado ✓


---
## 2e. MonoViT — Self-Supervised Monocular Depth with Vision Transformer (Zhao et al., 2022)

**MonoViT** reemplaza el encoder ResNet de Monodepth2 con **MPViT-Small** (*Multi-Path Vision Transformer*), un transformer que procesa parches de imagen en múltiples escalas simultáneamente. El decoder es `HR-Depth` (high-resolution decoder con attention modules).

Sus ventajas clave para este experimento:
- **Transformer encoder**: captura dependencias de largo rango en la imagen — potencialmente más robusto a variaciones de iluminación localizada (especulares) que los encoders convolucionales
- **Multi-path**: procesa la imagen a 4 escalas en paralelo, combinando contexto global y detalle local
- **Estado del arte** en KITTI (AbsRel=0.099 @ 640×192)

> Zhao, C., Zhang, Y., Poggi, M., Tosi, F., Guo, X., Zhu, Z., Huang, G., Tang, Y., & Mattoccia, S. (2022). MonoViT: Self-Supervised Monocular Depth Estimation with a Vision Transformer. *3DV 2022*. arXiv:2208.03543.

---
## 3. EndoLMSPEC — Arquitectura y carga del modelo

### 3.1 ¿Qué es EndoLMSPEC?

EndoLMSPEC (*Multi-Scale Structural-aware Exposure Correction for Endoscopic Imaging*, García-Vega et al., 2022) es una extensión del método LMSPEC de Afifi et al. (2021) adaptada específicamente para imágenes endoscópicas. Resuelve el problema de **corrección de exposición** (tanto sobreexposición por reflejos especulares como subexposición en zonas periféricas) sin datos pareados anotados manualmente — aprende a corregir usando la distribución estadística de imágenes endoscópicas normales.

Fue entrenado en **Endo4IE**, el primer dataset endoscópico diseñado específicamente para evaluación de métodos de image enhancement, que contiene frames reales y sintéticos con exposición incorrecta y sus correspondientes imágenes de referencia.

### 3.2 Arquitectura detallada

```mermaid
flowchart LR
    IN["Imagen de entrada\nHxWx3 float32"] --> LP

    subgraph LP [" 🔺 Pirámide Laplaciana (4 niveles) "]
        direction TB
        L4["Level 4\nH/8 × W/8\n(low-freq)"]
        L3["Level 3\nH/4 × W/4"]
        L2["Level 2\nH/2 × W/2"]
        L1["Level 1\nH × W\n(high-freq)"]
    end

    L4 --> U1

    subgraph UNETS [" 🧱 Cascada de U-Nets "]
        direction TB
        U1["UNet24\n(sin residual)\nCorrige low-freq"]
        U2["UNet24-res\n(+ residual)\nRefina freq. medias"]
        U3["UNet24-res\n(+ residual)\nRefina más detalle"]
        U4["UNet16-res\n(+ residual)\nReconstrucción final"]
    end

    U1 -->|"y_hat0 + L3"| U2
    U2 -->|"y_hat1 + L2"| U3
    U3 -->|"y_hat2 + L1"| U4

    U4 --> OUT["subnet_16\nImagen corregida\nHxWx3"]

    style LP fill:#fff8e1,stroke:#E0A800
    style UNETS fill:#e8f4fd,stroke:#2766CB
```

**Clave de la arquitectura**: la imagen de entrada se descompone en una **pirámide Laplaciana** de 4 niveles. El nivel 4 (más pequeño, baja frecuencia) captura la iluminación global; los niveles superiores capturan detalles de alta frecuencia (texturas, bordes, venas). Cada U-Net procesa un nivel de la pirámide y pasa su resultado al siguiente nivel sumando con los detalles de alta frecuencia. Esto permite corregir la iluminación global sin destruir la textura del tejido.

### 3.3 Pirámide Laplaciana

La pirámide Laplaciana descompone la imagen como:

$$L_k = G_k - \text{pyrUp}(G_{k+1})$$

donde $G_k$ es el nivel $k$ de la pirámide Gaussiana. Cada $L_k$ contiene las **diferencias** entre escalas consecutivas — esencialmente el contenido de alta frecuencia a esa escala. La imagen original se puede reconstruir exactamente sumando todos los niveles de la pirámide Laplaciana.

**¿Por qué es ideal para endoscopía?** Los reflejos especulares son eventos de alta frecuencia localizada (brillos puntuales), mientras que el gradiente radial de iluminación del endoscopio es de baja frecuencia. La pirámide Laplaciana los separa naturalmente, permitiendo que la red corrija la iluminación global (niveles bajos) sin alterar los detalles del tejido (niveles altos).

### 3.4 Función de pérdida

EndoLMSPEC usa una pérdida compuesta de 4 términos:

$$\mathcal{L} = \alpha \mathcal{L}_{rec} + \beta \mathcal{L}_{pyr} + \gamma \mathcal{L}_{SSIM} + \delta \mathcal{L}_{adv}$$

donde:
- $\mathcal{L}_{rec}$: reconstrucción pixel-wise (L1)
- $\mathcal{L}_{pyr}$: consistencia de pirámide Laplaciana
- $\mathcal{L}_{SSIM}$: similitud estructural (aportación principal vs. LMSPEC original)
- $\mathcal{L}_{adv}$: pérdida adversarial (discriminador)

La inclusión de $\mathcal{L}_{SSIM}$ es la innovación principal respecto al LMSPEC original: fuerza a la red a preservar la **estructura del tejido** (bordes, venas, pliegues) incluso al corregir la exposición — esencial para que la imagen corregida sea útil para estimación de profundidad.

In [ ]:
import sys
import subprocess
import torch

subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'path'])

import importlib.util, types

def _load_endolmspec(lmspec_path, device):
    """Carga Generator aislando su path para evitar conflictos."""
    _orig_path = sys.path.copy()
    # Excluir repos con modulos 'utils' que colisionan con EndoLMSPEC.
    clean = [str(lmspec_path)] + [
        p for p in sys.path
        if 'endoslam' not in p.lower() and 'endosfm' not in p.lower()
        and 'hadepth' not in p.lower()
        and 'endovit' not in p.lower()
        and 'experiments' not in p.lower()
    ]
    for _k in list(sys.modules.keys()):
        if (_k == 'utils' or _k.startswith('utils.')
                or _k in {'generator', 'unet'}):
            del sys.modules[_k]

    try:
        sys.path = clean
        spec = importlib.util.spec_from_file_location(
            'generator', lmspec_path / 'generator.py')
        mod = importlib.util.module_from_spec(spec)
        spec.loader.exec_module(mod)
        Generator = mod.Generator
    finally:
        sys.path = _orig_path

    net = Generator(n_channels=3, device=device, bilinear=False)
    return net, Generator

lmspec_net, _ = _load_endolmspec(LMSPEC_PATH, DEVICE)
lmspec_net.load_state_dict(torch.load(LMSPEC_WEIGHTS, map_location=DEVICE))
lmspec_net.to(DEVICE).eval()

n_params = sum(p.numel() for p in lmspec_net.parameters())
print(f'EndoLMSPEC parametros: {n_params/1e6:.2f} M')
print(f'Pesos: {LMSPEC_WEIGHTS.name}')
print('EndoLMSPEC cargado OK')


---
## 3b. IAT / EndoViT — Illumination-Adaptive Transformer fine-tuneado en Endo4IE

### ¿Qué es IAT y cómo se relaciona con EndoViT?

El repositorio **EndoViT** (proporcionado por el Dr. Gilberto Ochoa Ruiz) contiene una implementación del **IAT** (*Illumination-Adaptive Transformer*, Wang et al., 2022) con pesos fine-tuneados en el dataset **Endo4IE** — específicamente con una función de pérdida que combina pirámide Laplaciana e histograma GAN (`best_Epoch50_laplacian_histogan_loss.pth`). Esto lo distingue del IAT original entrenado en imágenes naturales (MIT-FiveK) y lo hace directamente comparable con EndoLMSPEC, que también fue entrenado en Endo4IE.

IAT opera en dos ramas paralelas:

- **Rama local** (`Local_pred_S`): predice dos mapas por píxel — un factor multiplicativo `mul` y un desplazamiento aditivo `add`: $\hat{I}_{local} = I \odot mul + add$
- **Rama global** (`Global_pred`): un Swin Transformer que predice gamma $\gamma$ y una matriz de corrección de color CCM 3×3

```mermaid
flowchart LR
    IN["Imagen\n[0,1] float\n(Endo4IE fine-tune)"] --> LP & GP

    subgraph LP [" 🔲 Rama Local "]
        LC["Conv + 3×CBlock_ln"] --> MUL["mul map\n(ReLU)"]
        LC --> ADD["add map\n(Tanh)"]
    end

    subgraph GP [" 🌐 Rama Global "]
        ST["Swin Transformer\n(Global_pred)"] --> GAMMA["γ (gamma)"]
        ST --> CCM["CCM 3×3\n(color matrix)"]
    end

    IN --> MULT["I × mul + add\n= I_local"]
    MUL & ADD --> MULT
    MULT --> APPLY["apply_color(I_local, CCM)^γ"]
    GAMMA & CCM --> APPLY
    APPLY --> OUT["Imagen corregida\n[0,1] float"]

    style LP fill:#e8f4fd,stroke:#2766CB
    style GP fill:#fff8e1,stroke:#E0A800
```

### Pesos utilizados

| Archivo | Descripción |
|---|---|
| `best_Epoch50_laplacian_histogan_loss.pth` | Fine-tune en Endo4IE (over+underexp), pérdida Laplaciana + HistoGAN, epoch 50 |

Otros checkpoints disponibles en `EndoVit/Endo4IE/`: epoch 10, 20 con variantes de normalización.

> Wang, T., Zhang, K., Shen, T., Luo, W., Stenger, B., & Lu, T. (2022). Ultra-High-Definition Low-Light Image Enhancement: A Benchmark and Transformer-Based Method. *AAAI 2022*.

In [ ]:
import sys
import types
import importlib.util
import torch

# global_net.py tiene `import imp` que fue removido en Python 3.12.
# imp no se usa en el código — solo está importado. Inyectamos un módulo
# ficticio para que el import no falle.
sys.modules['imp'] = types.ModuleType('imp')

# Importar IAT directamente por path para evitar conflicto con
# el módulo 'model' de Endo-Depth ya cargado en sys.path
iat_model_path = IAT_PATH / "experiments" / "model" / "IAT_main.py"
spec = importlib.util.spec_from_file_location("IAT_main", iat_model_path)
iat_module = importlib.util.module_from_spec(spec)

_iat_exp_path = str(IAT_PATH / "experiments")
if _iat_exp_path not in sys.path:
    sys.path.insert(0, _iat_exp_path)

spec.loader.exec_module(iat_module)
IAT = iat_module.IAT

# type='exp' — configuración usada en train_exposure.py
iat_net = IAT(in_dim=3, with_global=True, type='exp')
iat_net.load_state_dict(torch.load(IAT_WEIGHTS, map_location=DEVICE))
iat_net.to(DEVICE).eval()

n_params = sum(p.numel() for p in iat_net.parameters())
print(f"IAT parámetros : {n_params/1e3:.1f} K")
print(f"Pesos          : {IAT_WEIGHTS.name}")
print("IAT (EndoViT) cargado ✓")

---
## 4. Métodos de Image Enhancement

### 4.1 Baseline: sin corrección
La imagen original sin ningún preprocesamiento. Referencia para cuantificar el impacto de los métodos.

### 4.2 Retinex SSR
$$R_{SSR}(x,y) = \log I(x,y) - \log\left[G_\sigma * I(x,y)\right], \quad \sigma=30$$
> Rahman, Z., Jobson, D. J., & Woodell, G. A. (2004). *Journal of Electronic Imaging*, 13(1), 100–110. https://doi.org/10.1117/1.1636183

### 4.3 EndoLMSPEC
Pirámide Laplaciana de 4 niveles → cascada de 4 U-Nets con pérdida SSIM. Salida: `subnet_16` a resolución completa. Ver sección 3 para detalles.

### 4.4 IAT (Illumination-Adaptive Transformer)
Corrección local (mul/add por píxel vía conv) + corrección global (gamma + CCM vía Swin Transformer). ~90K parámetros, muy rápido. Ver sección 3b para detalles.

> Wang, T., et al. (2022). Ultra-High-Definition Low-Light Image Enhancement. *AAAI 2022*.

## 4. Métodos de realce de imagen (*enhancement*)

Comparamos **5 estrategias** de pre-procesamiento aplicadas **antes** de estimar profundidad:

| Método | Tipo | Idea |
|---|---|---|
| **none** | — | Imagen original (línea base) |
| **retinex** | Clásico | *Single-Scale Retinex*: homogeniza el brillo |
| **endolmspec** | *Deep* | EndoLMSPEC: pirámide Laplaciana + U-Nets |
| **iat** | *Deep* | IAT: corrige exposición (local + global) |
| **endosttn** | *Deep · temporal* | **Endo-STTN** (nuevo): elimina reflejos especulares |

### Endo-STTN — el método nuevo

A diferencia de los demás (una imagen aislada), Endo-STTN es **temporal**: trata la secuencia del *keyframe* como vídeo y rellena (*inpainting*) los **reflejos especulares** con lo que ven los **10 fotogramas vecinos**. Detecta los reflejos (máscara), un *transformer* espacio-temporal reconstruye el tejido oculto, y devuelve cada fotograma corregido. El *split* de 550 fotogramas **consecutivos** es justo lo que necesita esta vecindad temporal.

In [ ]:
import cv2
import numpy as np
import torch
import torchvision.transforms as T


def correct_none(img_rgb: np.ndarray) -> np.ndarray:
    return img_rgb


def correct_retinex(img_rgb: np.ndarray, sigma: float = 30) -> np.ndarray:
    """Single-Scale Retinex (Rahman et al., 2004)."""
    img_f = img_rgb.astype(np.float32) + 1.0
    result = np.zeros_like(img_f)
    for c in range(3):
        blur = cv2.GaussianBlur(img_f[:, :, c], (0, 0), sigma)
        result[:, :, c] = np.log(img_f[:, :, c]) - np.log(blur + 1.0)
    result -= result.min()
    return (result / (result.max() + 1e-8) * 255).astype(np.uint8)


def correct_endolmspec(img_rgb: np.ndarray, net, device) -> np.ndarray:
    """EndoLMSPEC: pirámide Laplaciana + U-Nets (García-Vega et al., 2022)."""
    img_t = T.ToTensor()(img_rgb).to(device)
    with torch.no_grad():
        _, outputs = net(img_t)
    out = outputs['subnet_16'][0].cpu().clamp(0, 1)
    return (out.permute(1, 2, 0).numpy() * 255).astype(np.uint8)


def correct_iat(img_rgb: np.ndarray, net, device) -> np.ndarray:
    """IAT: local mul/add + global gamma+CCM (Wang et al., 2022)."""
    # IAT espera [0,1] float, sin normalización adicional
    img_f = img_rgb.astype(np.float32) / 255.0
    img_t = torch.from_numpy(img_f).permute(2, 0, 1).unsqueeze(0).to(device)
    with torch.no_grad():
        _, _, enhanced = net(img_t)
    out = enhanced[0].cpu().clamp(0, 1)
    return (out.permute(1, 2, 0).numpy() * 255).astype(np.uint8)



# ---------------------------------------------------------------------
# Endo-STTN (Spatio-Temporal Transformer Network): quita reflejos
# especulares con informacion temporal. Es TEMPORAL: procesa la
# secuencia completa de un keyframe del split de una vez.
# ---------------------------------------------------------------------
import PIL.Image as pil

if IN_COLAB and not STTN_WEIGHTS.exists():
    STTN_CKPT_DIR.mkdir(parents=True, exist_ok=True)
    subprocess.check_call([sys.executable,"-m","pip","install","-q","gdown"])
    import gdown
    gdown.download(id=STTN_GDRIVE_ID, output=str(STTN_WEIGHTS), quiet=False)

def _load_endo_sttn(sttn_path, ckpt, device):
    _orig_path = sys.path.copy()
    _orig_mods = {k: sys.modules[k] for k in list(sys.modules)
                  if k in ("core","model") or k.startswith(("core.","model."))}
    for k in list(sys.modules):
        if k in ("core","model") or k.startswith(("core.","model.")): del sys.modules[k]
    try:
        sys.path.insert(0, str(sttn_path))
        net_mod = __import__("model.sttn", fromlist=["InpaintGenerator"])
        utils_mod = __import__("core.utils", fromlist=["Stack","ToTorchFormatTensor"])
        model = net_mod.InpaintGenerator().to(device)
        data = torch.load(ckpt, map_location=device); model.load_state_dict(data["netG"]); model.eval()
        Stack = utils_mod.Stack; ToTorch = utils_mod.ToTorchFormatTensor
    finally:
        sys.path = _orig_path
        for k in list(sys.modules):
            if k in ("core","model") or k.startswith(("core.","model.")): del sys.modules[k]
        sys.modules.update(_orig_mods)
    return model, Stack, ToTorch

STTN_W, STTN_H = 288, 288
STTN_REF_LEN, STTN_STRIDE = 10, 5
_sttn_ok = STTN_WEIGHTS.exists()
if _sttn_ok:
    sttn_model, _Stack, _ToTorch = _load_endo_sttn(STTN_PATH, STTN_WEIGHTS, DEVICE)
    from torchvision import transforms as _tvt
    _sttn_to_tensors = _tvt.Compose([_Stack(), _ToTorch()])
    print("Endo-STTN OK")
else:
    print("Endo-STTN: pesos no encontrados, se omitira")

def _sttn_specular_mask(img_rgb, dil=8):
    L = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2LAB)[:,:,0].astype(np.float32)
    m = (L >= np.percentile(L, 97)).astype(np.uint8)
    if dil: m = cv2.dilate(m, cv2.getStructuringElement(cv2.MORPH_ELLIPSE,(dil,dil)), iterations=1)
    return m

def _sttn_ref_idx(neighbor_ids, length):
    return [i for i in range(0, length, STTN_REF_LEN) if i not in neighbor_ids]

@torch.no_grad()
def endo_sttn_inpaint_sequence(frames_rgb):
    if not _sttn_ok: return frames_rgb
    H0, W0 = frames_rgb[0].shape[:2]
    pil_frames = [pil.fromarray(f).resize((STTN_W,STTN_H), pil.LANCZOS) for f in frames_rgb]
    masks_np   = [cv2.resize(_sttn_specular_mask(f),(STTN_W,STTN_H),interpolation=cv2.INTER_NEAREST) for f in frames_rgb]
    pil_masks  = [pil.fromarray((m*255).astype(np.uint8)) for m in masks_np]
    vlen = len(pil_frames)
    feats = _sttn_to_tensors(pil_frames).unsqueeze(0)*2-1
    masks = _sttn_to_tensors(pil_masks).unsqueeze(0)
    feats, masks = feats.to(DEVICE), masks.to(DEVICE)
    bin_masks = [np.expand_dims((np.array(m)!=0).astype(np.uint8),2) for m in pil_masks]
    raw = [np.array(f).astype(np.uint8) for f in pil_frames]
    feats = sttn_model.encoder((feats*(1-masks).float()).view(vlen,3,STTN_H,STTN_W))
    _, c, fh, fw = feats.size(); feats = feats.view(1,vlen,c,fh,fw)
    comp = [None]*vlen
    for f in range(0, vlen, STTN_STRIDE):
        nb_ids = [i for i in range(max(0,f-STTN_STRIDE), min(vlen,f+STTN_STRIDE+1))]
        ref_ids = _sttn_ref_idx(nb_ids, vlen)
        pred_feat = sttn_model.infer(feats[0, nb_ids+ref_ids], masks[0, nb_ids+ref_ids])
        pred_img = torch.tanh(sttn_model.decoder(pred_feat[:len(nb_ids)]))
        pred_img = ((pred_img+1)/2).cpu().permute(0,2,3,1).numpy()*255
        for i, idx in enumerate(nb_ids):
            img = pred_img[i].astype(np.uint8)*bin_masks[idx] + raw[idx]*(1-bin_masks[idx])
            comp[idx] = img if comp[idx] is None else (comp[idx]*0.5+img*0.5).astype(np.uint8)
    return [cv2.resize(cc,(W0,H0),interpolation=cv2.INTER_LANCZOS4) for cc in comp]

_STTN_CACHE = {}
def correct_endo_sttn(img_rgb, ds=None, kf=None, fid=None):
    if not _sttn_ok: return img_rgb
    if ds is None: return endo_sttn_inpaint_sequence([img_rgb])[0]
    key = (ds, kf)
    if key not in _STTN_CACHE:
        fids = SPLIT_BY_KF[key]
        seq = [load_split_frame(ds, kf, f)[0] for f in fids]
        out = endo_sttn_inpaint_sequence(seq)
        _STTN_CACHE.clear()
        _STTN_CACHE[key] = {f:o for f,o in zip(fids,out)}
    return _STTN_CACHE[key][fid]

CORRECTIONS = {
    "none":       lambda img: correct_none(img),
    "retinex":    lambda img: correct_retinex(img),
    "endolmspec": lambda img: correct_endolmspec(img, lmspec_net, DEVICE),
    "iat":        lambda img: correct_iat(img, iat_net, DEVICE),
    "endosttn":   correct_endo_sttn,
}
print("Funciones de corrección definidas")
print(f"Métodos: {list(CORRECTIONS.keys())}")


---
## 5. Carga de datos SCARED

Mismo protocolo que Avance 3: datasets 8–9 (split de test, animal distinto al de train). Cada keyframe contiene `Left_Image.png` y `left_depth_map.tiff` (canal Z en mm).

## 3. Modelos de profundidad y métricas

### Los 5 modelos (pesos **originales** de cada artículo)

| Modelo | Arquitectura |
|---|---|
| **EndoDepth / Monodepth2** | ResNet-18 + *DepthDecoder* |
| **EndoSfMLearner** | DispResNet-18 |
| **MonoViT** | MPViT-Small + decodificador Monodepth2 |
| **AF-SfMLearner** | ResNet-18 + *Appearance Flow* |
| **HADepth** | DINOv2 + DoRA + *DPT Head* |

Predicen **profundidad relativa**; se alinea con el GT en mm vía ***median scaling***.

### Métricas (sobre píxeles válidos, `(0, 150] mm`)

| Métrica | Mide | Mejor |
|---|---|---|
| **AbsRel / SqRel** | Error relativo (abs / cuadrático) | ↓ |
| **RMSE / RMSELog** | Error cuadrático (lineal / log, mm) | ↓ |
| **δ < 1.25ᵏ** | Fracción de píxeles casi correctos (k=1,2,3) | ↑ |
| **Chamfer** | Distancia entre nubes de puntos 3D | ↓ |
| **AbsRel_spec / _nospec** | AbsRel dentro / fuera de reflejos | ↓ |

> La separación **spec/nospec** permite medir si Endo-STTN mejora la profundidad **justo en las zonas de reflejo**.

In [ ]:
import time
import torch
import torch.nn.functional as F
import numpy as np
import cv2
import PIL.Image as pil
from torchvision import transforms
from scipy.spatial import cKDTree
from skimage.metrics import structural_similarity as ssim_fn
from skimage.metrics import peak_signal_noise_ratio as psnr_fn
from skimage.transform import resize as imresize

FX, FY = 1078.0, 1078.0
CX, CY = 640.0,  512.0


def predict_depth_endodepth(img_rgb, encoder, decoder, feed_h, feed_w, device):
    H, W = img_rgb.shape[:2]
    input_t = transforms.ToTensor()(
        pil.fromarray(img_rgb).resize((feed_w, feed_h), pil.LANCZOS)
    ).unsqueeze(0).to(device)
    if device.type == "cuda": torch.cuda.synchronize()
    t0 = time.perf_counter()
    with torch.no_grad():
        features = encoder(input_t)
        outputs  = decoder(features)
    if device.type == "cuda": torch.cuda.synchronize()
    t_ms = (time.perf_counter() - t0) * 1000
    disp = F.interpolate(outputs[("disp", 0)], (H, W), mode="bilinear", align_corners=False)
    disp_np = disp.squeeze().cpu().numpy()
    min_d, max_d = 1/100, 1/0.1
    return 1.0 / (min_d + (max_d - min_d) * disp_np), t_ms


def predict_depth_endosfm(img_rgb, net, device, img_h=256, img_w=832):
    H, W = img_rgb.shape[:2]
    img_r = imresize(img_rgb, (img_h, img_w)).astype(np.float32)
    img_t = torch.from_numpy(
        ((img_r / 255.0 - 0.45) / 0.225).transpose(2, 0, 1)
    ).unsqueeze(0).to(device)
    if device.type == "cuda": torch.cuda.synchronize()
    t0 = time.perf_counter()
    with torch.no_grad():
        disp = net(img_t)
    if device.type == "cuda": torch.cuda.synchronize()
    t_ms = (time.perf_counter() - t0) * 1000
    disp_full = imresize(disp.squeeze().cpu().numpy(), (H, W))
    return 1.0 / (disp_full + 1e-6), t_ms


def predict_depth_monovit(img_rgb, net, feed_h, feed_w, device):
    H, W = img_rgb.shape[:2]
    input_t = transforms.ToTensor()(
        pil.fromarray(img_rgb).resize((feed_w, feed_h), pil.LANCZOS)
    ).unsqueeze(0).to(device)
    mean = torch.tensor([0.485, 0.456, 0.406]).view(1,3,1,1).to(device)
    std  = torch.tensor([0.229, 0.224, 0.225]).view(1,3,1,1).to(device)
    input_t = (input_t - mean) / std
    if device.type == "cuda": torch.cuda.synchronize()
    t0 = time.perf_counter()
    with torch.no_grad():
        outputs = net(input_t)
    if device.type == "cuda": torch.cuda.synchronize()
    t_ms = (time.perf_counter() - t0) * 1000
    disp = F.interpolate(outputs[("disp", 0)], (H, W), mode="bilinear", align_corners=False)
    disp_np = disp.squeeze().cpu().numpy()
    min_d, max_d = 1/100, 1/0.1
    return 1.0 / (min_d + (max_d - min_d) * disp_np), t_ms


def predict_depth_afsfm(img_rgb, encoder, decoder, feed_h, feed_w, device):
    H, W = img_rgb.shape[:2]
    input_t = transforms.ToTensor()(
        pil.fromarray(img_rgb).resize((feed_w, feed_h), pil.LANCZOS)
    ).unsqueeze(0).to(device)
    if device.type == "cuda": torch.cuda.synchronize()
    t0 = time.perf_counter()
    with torch.no_grad():
        features = encoder(input_t)
        outputs  = decoder(features)
    if device.type == "cuda": torch.cuda.synchronize()
    t_ms = (time.perf_counter() - t0) * 1000
    disp = F.interpolate(outputs[("disp", 0)], (H, W), mode="bilinear", align_corners=False)
    disp_np = disp.squeeze().cpu().numpy()
    min_d, max_d = 1/100, 1/0.1
    return 1.0 / (min_d + (max_d - min_d) * disp_np), t_ms


def predict_depth_hadepth(img_rgb, net, device):
    H, W = img_rgb.shape[:2]
    input_t = transforms.ToTensor()(pil.fromarray(img_rgb)).unsqueeze(0).to(device)
    mean = torch.tensor([0.485, 0.456, 0.406]).view(1,3,1,1).to(device)
    std  = torch.tensor([0.229, 0.224, 0.225]).view(1,3,1,1).to(device)
    input_t = (input_t - mean) / std
    if device.type == "cuda": torch.cuda.synchronize()
    t0 = time.perf_counter()
    with torch.no_grad():
        outputs = net(input_t)
    if device.type == "cuda": torch.cuda.synchronize()
    t_ms = (time.perf_counter() - t0) * 1000
    disp = F.interpolate(outputs[("disp", 0)], (H, W), mode="bilinear", align_corners=False)
    disp_np = disp.squeeze().cpu().numpy()
    min_d, max_d = 1/100, 1/0.1
    return 1.0 / (min_d + (max_d - min_d) * disp_np), t_ms


try:
    _ = monovit_net;  monovit_ok = True
except NameError:
    monovit_ok = False;  print("⚠️  MonoViT NO cargado")

try:
    _ = afsfm_encoder;  afsfm_ok = True
except NameError:
    afsfm_ok = False;  print("⚠️  AF-SfMLearner NO cargado")

try:
    _ = hadepth_net;  hadepth_ok = True
except NameError:
    hadepth_ok = False;  print("⚠️  HADepth NO cargado")

DEPTH_MODELS = {
    "EndoDepth":      lambda img: predict_depth_endodepth(
                          img, encoder, depth_decoder, FEED_HEIGHT, FEED_WIDTH, DEVICE),
    "EndoSfMLearner": lambda img: predict_depth_endosfm(img, endosfm_net, DEVICE),
}
if monovit_ok:
    DEPTH_MODELS["MonoViT"] = lambda img: predict_depth_monovit(
                                   img, monovit_net, MONOVIT_H, MONOVIT_W, DEVICE)
if afsfm_ok:
    DEPTH_MODELS["AF-SfMLearner"] = lambda img: predict_depth_afsfm(
                                        img, afsfm_encoder, afsfm_decoder,
                                        AFSFM_H, AFSFM_W, DEVICE)
if hadepth_ok:
    DEPTH_MODELS["HADepth"] = lambda img: predict_depth_hadepth(img, hadepth_net, DEVICE)


def depth_to_pc(depth_mm, mask, fx=FX, fy=FY, cx=CX, cy=CY):
    H, W = depth_mm.shape
    uu, vv = np.meshgrid(np.arange(W), np.arange(H))
    Z = depth_mm[mask]
    return np.stack([(uu[mask]-cx)*Z/fx, (vv[mask]-cy)*Z/fy, Z], axis=1)

def chamfer(pc1, pc2, max_pts=50_000):
    if len(pc1) == 0 or len(pc2) == 0: return np.nan
    rng = np.random.default_rng(42)
    if len(pc1) > max_pts: pc1 = pc1[rng.choice(len(pc1), max_pts, replace=False)]
    if len(pc2) > max_pts: pc2 = pc2[rng.choice(len(pc2), max_pts, replace=False)]
    d1, _ = cKDTree(pc2).query(pc1)
    d2, _ = cKDTree(pc1).query(pc2)
    return float((d1.mean() + d2.mean()) / 2)

def specular_mask(img_rgb, pct=97, dil=15):
    L = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2LAB)[:,:,0].astype(np.float32)
    mask = (L >= np.percentile(L, pct)).astype(np.uint8)
    return cv2.dilate(mask, cv2.getStructuringElement(cv2.MORPH_ELLIPSE,(dil,dil))).astype(bool)

def compute_metrics(img_orig, img_corr, depth_rel, gt_mm, cap_mm=150.0):
    valid = (~np.isnan(gt_mm)) & (gt_mm > 0) & (gt_mm < cap_mm)
    if valid.sum() == 0:
        nan_keys = ["AbsRel","SqRel","RMSE","RMSELog",
                    "delta_1","delta_2","delta_3",
                    "PSNR","SSIM","AbsRel_spec","AbsRel_nospec","scale"]
        return {k: np.nan for k in nan_keys}
    scale = np.median(gt_mm[valid]) / (np.median(depth_rel[valid]) + 1e-8)
    pred  = depth_rel * scale
    d, gt = pred[valid], gt_mm[valid]
    spec  = specular_mask(img_orig)
    vs, vn = valid & spec, valid & ~spec

    # Threshold accuracy: max(d/gt, gt/d) < thr
    ratio = np.maximum(d / (gt + 1e-8), gt / (d + 1e-8))
    return {
        "scale":        round(float(scale), 4),
        "AbsRel":       round(float(np.mean(np.abs(d-gt) / (gt+1e-8))), 4),
        "SqRel":        round(float(np.mean((d-gt)**2   / (gt+1e-8))), 4),
        "RMSE":         round(float(np.sqrt(np.mean((d-gt)**2))), 3),
        "RMSELog":      round(float(np.sqrt(np.mean(
                            (np.log(np.clip(d,1e-3,None)) -
                             np.log(np.clip(gt,1e-3,None)))**2))), 4),
        "delta_1":      round(float(np.mean(ratio < 1.25)),    4),
        "delta_2":      round(float(np.mean(ratio < 1.25**2)), 4),
        "delta_3":      round(float(np.mean(ratio < 1.25**3)), 4),
        "Chamfer":      round(chamfer(depth_to_pc(np.where(valid,pred,np.nan),valid),
                                      depth_to_pc(np.where(valid,gt_mm,np.nan),valid)), 3),
        "PSNR":         round(float(psnr_fn(img_orig, img_corr, data_range=255)), 2),
        "SSIM":         round(float(ssim_fn(img_orig, img_corr,
                                            channel_axis=2, data_range=255)), 4),
        "AbsRel_spec":  round(float(np.mean(np.abs(pred[vs]-gt_mm[vs])/(gt_mm[vs]+1e-8))),4)
                        if vs.sum()>0 else np.nan,
        "AbsRel_nospec":round(float(np.mean(np.abs(pred[vn]-gt_mm[vn])/(gt_mm[vn]+1e-8))),4)
                        if vn.sum()>0 else np.nan,
    }

print(f"Modelos cargados : {list(DEPTH_MODELS.keys())}")
print(f"Enhancements     : {list(CORRECTIONS.keys())}")
print(f"Total eval       : {len(DEPTH_MODELS)*len(CORRECTIONS)*len(SPLIT_ITEMS)}")
print("Metricas         : AbsRel, SqRel, RMSE, RMSELog, delta_1/2/3, Chamfer, AbsRel_spec")


---
## 6. Visualización comparativa de los métodos de enhancement

Antes de correr el experimento completo, comparamos visualmente los tres métodos sobre el keyframe de prueba. Esto permite verificar que:

1. EndoLMSPEC corrige correctamente la exposición
2. Retinex elimina el gradiente de iluminación
3. Ambos preservan la textura del tejido

---
## 7. Experimento factorial: 4 enhancements × 5 modelos × 10 keyframes

Total: **200 evaluaciones**.

In [ ]:
import numpy as np, cv2

def load_split_frame(ds, kf, fid):
    """Carga imagen RGB (uint8) y GT (mm, nan en invalidos) del caché en memoria."""
    img, gt = SPLIT_DATA[_key(ds, kf, fid)]
    return img, gt

# Frames consecutivos del mismo keyframe (para Endo-STTN, que es temporal)
from collections import defaultdict
SPLIT_BY_KF = defaultdict(list)
for ds, kf, fid in SPLIT_ITEMS:
    if _key(ds, kf, fid) in SPLIT_DATA:
        SPLIT_BY_KF[(ds, kf)].append(fid)
for k in SPLIT_BY_KF: SPLIT_BY_KF[k] = sorted(SPLIT_BY_KF[k])

_ds0, _kf0, _fid0 = SPLIT_ITEMS[0]
_img, _gt = load_split_frame(_ds0, _kf0, _fid0)
print(f"Ejemplo {_ds0}/{_kf0} frame {_fid0}: img {_img.shape}  "
      f"GT valido {(~np.isnan(_gt)).mean()*100:.1f}%" if _gt is not None else "sin GT")


## 5. Bucle de evaluación con *checkpointing*

Recorre **550 fotogramas × 5 *enhancements* × 5 modelos**. Guarda el CSV **cada 50 evaluaciones** y **reanuda** si la sesión se interrumpe (solo calcula lo que falta). El *enhancement* `endosttn` se invoca con `dataset/keyframe/frame` para usar su contexto temporal.

In [ ]:
import pandas as pd, numpy as np, time, os
from tqdm.notebook import tqdm

assert "MonoViT" in DEPTH_MODELS and "AF-SfMLearner" in DEPTH_MODELS and "HADepth" in DEPTH_MODELS
assert len(DEPTH_MODELS) == 5, f"Se esperan 5 modelos, hay {len(DEPTH_MODELS)}"
print(f"Modelos: {list(DEPTH_MODELS.keys())}")
print(f"Enhancements: {list(CORRECTIONS.keys())}")

CKPT_CSV = OUT_DIR / "avance4_newversion_results.csv"
CKPT_EVERY = 50

if CKPT_CSV.exists():
    df_prev = pd.read_csv(CKPT_CSV)
    done = set(zip(df_prev["Dataset"],df_prev["Keyframe"],df_prev["Frame"],df_prev["Método"],df_prev["Modelo"]))
    results = df_prev.to_dict("records")
    print(f"Reanudando: {len(done)} evaluaciones ya hechas")
else:
    done, results = set(), []

def _apply_correction(name, fn, img_rgb, ds, kf, fid):
    if name == "endosttn": return fn(img_rgb, ds=ds, kf=kf, fid=fid)
    return fn(img_rgb)

n_new = 0
for (ds, kf, fid) in tqdm(SPLIT_ITEMS, desc="Frames del split"):
    img_rgb, gt_mm = load_split_frame(ds, kf, fid)
    if gt_mm is None: continue
    for corr_name, corr_fn in CORRECTIONS.items():
        t0 = time.perf_counter()
        img_corr = _apply_correction(corr_name, corr_fn, img_rgb, ds, kf, fid)
        t_enh = (time.perf_counter()-t0)*1000
        for model_name, depth_fn in DEPTH_MODELS.items():
            if (ds,kf,fid,corr_name,model_name) in done: continue
            depth_rel, t_inf = depth_fn(img_corr)
            m = compute_metrics(img_rgb, img_corr, depth_rel, gt_mm, CAP_MM)
            results.append({
                "Dataset":ds,"Keyframe":kf,"Frame":fid,
                "Modelo":model_name,"Método":corr_name,
                "T_enhance_ms":round(t_enh,1),"T_depth_ms":round(t_inf,1),
                "T_total_ms":round(t_enh+t_inf,1),
                "FPS":round(1000/(t_enh+t_inf+1e-6),1),
                **m,
            })
            n_new += 1
            if n_new % CKPT_EVERY == 0:
                pd.DataFrame(results).to_csv(CKPT_CSV, index=False)

df = pd.DataFrame(results)
df.to_csv(CKPT_CSV, index=False)
print(f"\nExperimento completo - {len(df)} evaluaciones ({n_new} nuevas)")


In [ ]:
import pandas as pd
import numpy as np

numeric_cols = ["AbsRel","SqRel","RMSE","RMSELog",
                "delta_1","delta_2","delta_3",
                "AbsRel_spec","AbsRel_nospec","T_total_ms","FPS"]

summary = (
    df.groupby(["Modelo","Método"])[numeric_cols]
    .mean().round(4)
    .sort_values(["Modelo","AbsRel"])
)

# Mejora vs baseline (none) por modelo
for model in df["Modelo"].unique():
    sub = summary.loc[model]
    if "none" in sub.index:
        base_ar   = sub.loc["none","AbsRel"]
        base_spec = sub.loc["none","AbsRel_spec"]
        summary.loc[(model,slice(None)),"DeltaAbsRel_%"] = (
            (summary.loc[(model,slice(None)),"AbsRel"] - base_ar)/base_ar*100).round(1)
        summary.loc[(model,slice(None)),"DeltaSpec_%"] = (
            (summary.loc[(model,slice(None)),"AbsRel_spec"] - base_spec)/base_spec*100).round(1)

# ── Tabla estilo paper (baseline=none por modelo) ──────────────────────────
print("=" * 110)
print(f"{'Modelo':<18} {'Enhancement':<12} {'AbsRel':>7} {'SqRel':>7} {'RMSE':>7} "
      f"{'RMSELog':>8} {'d<1.25':>7} {'d<1.25^2':>9} {'d<1.25^3':>9} "
      f"{'IT(ms)':>7} {'DeltaAR%':>9}")
print("-" * 110)

for model in df["Modelo"].unique():
    for method in ["none","retinex","endolmspec","iat","endosttn"]:
        if (model,method) not in summary.index: continue
        row = summary.loc[(model,method)]
        delta = f"{row['DeltaAbsRel_%']:+.1f}%" if method != "none" else "baseline"
        marker = " ←" if method != "none" and row["AbsRel"] < summary.loc[(model,"none"),"AbsRel"] else ""
        print(f"  {model:<16} {method:<12} "
              f"{row['AbsRel']:>7.4f} {row['SqRel']:>7.4f} {row['RMSE']:>7.3f} "
              f"{row['RMSELog']:>8.4f} {row['delta_1']:>7.4f} {row['delta_2']:>9.4f} "
              f"{row['delta_3']:>9.4f} {row['T_total_ms']:>7.1f} {delta:>9}{marker}")
    print()

print("d<1.25^k = fraccion de pixeles con max(pred/gt, gt/pred) < 1.25^k (mas alto = mejor)")
print("DeltaAR% negativo = improvement vs baseline (none)")


In [ ]:
# Asegurar que FPS esta en summary (compatibilidad con versiones anteriores)
if 'FPS' not in summary.columns:
    summary['FPS'] = df.groupby(['Modelo','Método'])['FPS'].mean().round(1)
    print('FPS agregado al summary')


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

models_list  = list(df["Modelo"].unique())
methods_list = list(df["Método"].unique())
colors = {"none":"#888888", "retinex":"#2766CB", "endolmspec":"#E0A800", "iat":"#2ecc71"}

metrics_plot = [
    ("AbsRel",       "AbsRel (↓ mejor)",       "Geométrica — primaria"),
    ("RMSE",         "RMSE mm (↓ mejor)",        "Geométrica"),
    ("AbsRel_spec",  "AbsRel especular (↓ mejor)","Hipótesis central"),
    ("FPS",          "FPS total (↑ mejor)",      "Viabilidad clínica"),
]

n_metrics = len(metrics_plot)
n_models  = len(models_list)
x = np.arange(len(methods_list))
bar_w = 0.6

fig, axes = plt.subplots(n_models, n_metrics,
                         figsize=(5 * n_metrics, 4.5 * n_models),
                         sharey="col")   # mismo eje Y por columna (métrica)

fig.suptitle("Evaluación: Image Enhancement × Modelos de Depth en SCARED\n"
             "(promedio 10 keyframes, datasets 8–9)",
             fontsize=14, fontweight="bold", y=1.01)

for row, model in enumerate(models_list):
    sub = summary.loc[model] if model in summary.index.get_level_values(0) else None

    for col, (metric, label, cat) in enumerate(metrics_plot):
        ax = axes[row, col] if n_models > 1 else axes[col]

        vals = []
        for m in methods_list:
            if sub is not None and m in sub.index:
                vals.append(sub.loc[m, metric])
            else:
                vals.append(np.nan)

        bar_colors = [colors.get(m, "#aaa") for m in methods_list]
        bars = ax.bar(x, vals, width=bar_w, color=bar_colors, edgecolor="white", linewidth=0.8)

        ax.set_xticks(x)
        ax.set_xticklabels(methods_list, fontsize=11)
        ax.grid(axis="y", alpha=0.3)

        # Título de columna solo en primera fila
        if row == 0:
            ax.set_title(f"{label}\n[{cat}]", fontsize=10, pad=6)

        # Etiqueta de modelo solo en primera columna
        if col == 0:
            ax.set_ylabel(model, fontsize=12, fontweight="bold", labelpad=8)

        # Valores encima de cada barra
        for bar, val in zip(bars, vals):
            if not np.isnan(val):
                ax.text(bar.get_x() + bar.get_width() / 2,
                        bar.get_height() + ax.get_ylim()[1] * 0.01,
                        f"{val:.3f}",
                        ha="center", va="bottom", fontsize=9, fontweight="bold")

plt.tight_layout()
plt.savefig(OUT_DIR / "avance4_metricas_comparativa.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

methods_list = list(df["Método"].unique())
models_list  = list(df["Modelo"].unique())
x = np.arange(len(methods_list))
w = 0.3

# Un panel por modelo × 2 vistas (keyframe_0 y promedio)
n_cols = 2
n_rows = len(models_list)
fig, axes = plt.subplots(n_rows, n_cols, figsize=(14, 5 * n_rows), sharey="row")
if n_rows == 1:
    axes = axes[np.newaxis, :]

fig.suptitle("Robustez a especulares: AbsRel en zona especular vs. tejido normal",
             fontsize=13, fontweight="bold")

_kf0 = list(SPLIT_BY_KF.keys())[0]   # primer keyframe del split
panels = [
    (_kf0[0], _kf0[1]),
    (None, None),
]
panel_titles = [f"{_kf0[0]}/{_kf0[1]}", "Promedio — todos los frames del split"]

for row, model in enumerate(models_list):
    for col, ((ds_id, kf_id), ptitle) in enumerate(zip(panels, panel_titles)):
        ax = axes[row, col]

        if ds_id is not None:
            sub = df[(df["Modelo"]==model) & (df["Dataset"]==ds_id) & (df["Keyframe"]==kf_id)]
        else:
            sub = df[df["Modelo"]==model]

        spec_vals   = [sub[sub["Método"]==m]["AbsRel_spec"].mean()   for m in methods_list]
        nospec_vals = [sub[sub["Método"]==m]["AbsRel_nospec"].mean() for m in methods_list]

        b1 = ax.bar(x - w/2, spec_vals,   w, label="Zona especular",  color="#e74c3c", alpha=0.85)
        b2 = ax.bar(x + w/2, nospec_vals, w, label="Tejido normal",   color="#2766CB", alpha=0.85)

        ax.set_xticks(x)
        ax.set_xticklabels(methods_list, fontsize=11)
        ax.set_ylabel("AbsRel")
        ax.set_title(f"{model} — {ptitle}", fontsize=10)
        ax.legend(fontsize=9)
        ax.grid(axis="y", alpha=0.3)

        for bars, vals in [(b1, spec_vals), (b2, nospec_vals)]:
            for bar, val in zip(bars, vals):
                if not np.isnan(val):
                    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height(),
                            f"{val:.3f}", ha="center", va="bottom", fontsize=8)

plt.tight_layout()
plt.savefig(OUT_DIR / "avance4_robustez_especular.png", dpi=150, bbox_inches="tight")
plt.show()

# Interpretación cuantitativa por modelo
for model in models_list:
    sub_none = df[(df["Modelo"]==model) & (df["Método"]=="none")]
    if sub_none.empty:
        continue
    b_spec = sub_none["AbsRel_spec"].mean()
    b_nosp = sub_none["AbsRel_nospec"].mean()
    print(f"\n▶ {model} — reducción vs. baseline (none):")
    for m in methods_list:
        if m == "none": continue
        sub_m = df[(df["Modelo"]==model) & (df["Método"]==m)]
        s = sub_m["AbsRel_spec"].mean()
        n = sub_m["AbsRel_nospec"].mean()
        print(f"  {m:12s}: spec {s:.4f} ({(b_spec-s)/b_spec*100:+.1f}%)  "
              f"nospec {n:.4f} ({(b_nosp-n)/b_nosp*100:+.1f}%)")

## 6. Visualización cualitativa

Mapas de profundidad predichos por cada modelo bajo cada *enhancement*, junto al *ground-truth*, para dos fotogramas representativos del *split*. Color `plasma_r`: **amarillo = cerca, azul = lejos** (normalizado p2–p98).

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

_kf_list = list(SPLIT_BY_KF.keys())
VIZ_FRAMES = [(_kf_list[0][0], _kf_list[0][1], SPLIT_BY_KF[_kf_list[0]][0]),
              (_kf_list[len(_kf_list)//2][0], _kf_list[len(_kf_list)//2][1], SPLIT_BY_KF[_kf_list[len(_kf_list)//2]][0])]
CMAP_DEPTH = 'plasma_r'  # cerca=amarillo brillante, lejos=azul oscuro

for ds_id, kf_id, fid in VIZ_FRAMES:
    img_rgb, gt_mm = load_split_frame(ds_id, kf_id, fid)
    valid_gt = (~np.isnan(gt_mm)) & (gt_mm > 0) & (gt_mm < CAP_MM)
    n_corr   = len(CORRECTIONS)
    n_models = len(DEPTH_MODELS)
    corr_names = list(CORRECTIONS.keys())

    # Pre-calcular predicciones
    cache = {}
    for model_name, depth_fn in DEPTH_MODELS.items():
        for corr_name, corr_fn in CORRECTIONS.items():
            img_c = corr_fn(img_rgb, ds=ds_id, kf=kf_id, fid=fid) if corr_name=='endosttn' else corr_fn(img_rgb)
            depth_rel, _ = depth_fn(img_c)
            scale = np.median(gt_mm[valid_gt]) / (np.median(depth_rel[valid_gt]) + 1e-8)
            pred_mm = depth_rel * scale
            cache[(model_name, corr_name)] = (img_c, depth_rel, pred_mm)

    # Layout: n_models filas x (1 RGB + n_corr depth + 1 GT) columnas
    n_cols = 1 + n_corr + 1
    fig, axes = plt.subplots(n_models, n_cols,
                             figsize=(3.5 * n_cols, 3.5 * n_models))

    for row, (model_name, depth_fn) in enumerate(DEPTH_MODELS.items()):
        ax_row = axes[row] if n_models > 1 else axes

        # Col 0: imagen RGB
        ax_row[0].imshow(img_rgb)
        ax_row[0].axis('off')
        ax_row[0].set_ylabel(model_name, fontsize=9, fontweight='bold')
        if row == 0:
            ax_row[0].set_title('RGB', fontsize=9)

        # Cols 1..n_corr: depth maps, normalizado por imagen
        for col, corr_name in enumerate(corr_names, start=1):
            img_c, depth_rel, pred_mm = cache[(model_name, corr_name)]
            m = compute_metrics(img_c, img_c, depth_rel, gt_mm, CAP_MM)
            vmin_i = np.nanpercentile(pred_mm[valid_gt], 2)
            vmax_i = np.nanpercentile(pred_mm[valid_gt], 98)
            ax_row[col].imshow(pred_mm, cmap=CMAP_DEPTH, vmin=vmin_i, vmax=vmax_i)
            ax_row[col].axis('off')
            if row == 0:
                ax_row[col].set_title(f'{corr_name}\nAbsRel={m["AbsRel"]:.4f}', fontsize=8)
            else:
                ax_row[col].set_title(f'AbsRel={m["AbsRel"]:.4f}', fontsize=8)

        # Ultima col: GT
        vmin_gt = np.nanpercentile(gt_mm[valid_gt], 2)
        vmax_gt = np.nanpercentile(gt_mm[valid_gt], 98)
        ax_row[-1].imshow(gt_mm, cmap=CMAP_DEPTH, vmin=vmin_gt, vmax=vmax_gt)
        ax_row[-1].axis('off')
        if row == 0:
            ax_row[-1].set_title('GT (mm)', fontsize=9)

    plt.suptitle(f'NV {ds_id}/{kf_id} f{fid}  —  plasma_r: amarillo=cerca, azul=lejos',
                 fontsize=10, fontweight='bold', y=1.01)
    plt.tight_layout()
    plt.savefig(OUT_DIR / f'avance4nv_viz_{ds_id}_{kf_id}_{fid}.png',
                dpi=120, bbox_inches='tight')
    plt.show()
    print(f'Guardado: avance4nv_viz_{ds_id}_{kf_id}_{fid}.png')


---
## 11. Conclusiones

### Tabla completa de resultados (promedio 10 keyframes, datasets 8–9 SCARED)

| Modelo | Enhancement | AbsRel↓ | SqRel↓ | RMSE↓ | RMSELog↓ | δ<1.25↑ | δ<1.25²↑ | δ<1.25³↑ | IT (ms) | ΔAR% |
|---|---|---|---|---|---|---|---|---|---|---|
| **Endo-Depth** | none | 0.1652 | 2.6842 | 12.468 | 0.2119 | 0.7764 | 0.9420 | 0.9900 | 27.4 | baseline |
| | retinex | 0.1533 | 2.3998 | 11.578 | 0.2137 | 0.8015 | 0.9394 | 0.9823 | 372.2 | **−7.2%** ← |
| | endolmspec | 0.1635 | 2.8055 | 12.949 | 0.2259 | 0.7660 | 0.9299 | 0.9848 | 143.6 | −1.0% ← |
| | **iat** | **0.1454** | **2.1133** | **11.074** | 0.1899 | **0.8223** | **0.9581** | 0.9910 | 271.4 | **−12.0%** ← |
| **EndoSfMLearner** | none | 0.2906 | 7.6026 | 21.002 | 0.3802 | 0.4706 | 0.7550 | 0.9148 | 6.2 | baseline |
| | retinex | 0.2908 | 7.6062 | 21.000 | 0.3804 | 0.4706 | 0.7548 | 0.9142 | 351.1 | +0.1% |
| | endolmspec | 0.2904 | 7.5904 | 20.981 | 0.3797 | 0.4709 | 0.7558 | 0.9154 | 122.4 | −0.1% ← |
| | iat | 0.2902 | 7.5804 | 20.969 | 0.3795 | 0.4712 | 0.7560 | 0.9157 | 250.4 | −0.1% ← |
| **MonoViT** | none | 0.3046 | 10.2427 | 23.729 | 0.3532 | 0.4838 | 0.7896 | 0.9348 | 53.3 | baseline |
| | retinex | 0.3139 | 11.5943 | 26.566 | 0.3438 | 0.4574 | 0.8143 | 0.9510 | 397.7 | +3.1% |
| | endolmspec | 0.3305 | 12.3400 | 26.422 | 0.3712 | 0.4422 | 0.7657 | 0.9273 | 169.2 | +8.5% |
| | **iat** | **0.2825** | **8.1560** | **21.340** | 0.3280 | **0.4917** | **0.8321** | 0.9485 | 297.0 | **−7.3%** ← |
| **AF-SfMLearner** | **none** | **0.1060** | **1.3628** | **8.815** | **0.1461** | **0.9025** | 0.9829 | 0.9912 | 27.4 | baseline |
| | retinex | 0.1271 | 1.6887 | 9.915 | 0.1709 | 0.8464 | 0.9798 | 0.9913 | 372.2 | +19.9% |
| | endolmspec | 0.1074 | 1.3426 | 8.586 | 0.1436 | 0.9104 | 0.9832 | 0.9916 | 143.4 | +1.3% |
| | iat | 0.1120 | 1.4800 | 9.078 | 0.1511 | 0.8923 | 0.9825 | 0.9912 | 271.6 | +5.7% |
| **HADepth** | **none** | **0.0444** | **0.2534** | **3.683** | **0.0592** | **0.9945** | **0.9990** | **0.9996** | 32.1 | baseline |
| | retinex | 0.0849 | 0.7427 | 6.333 | 0.1098 | 0.9195 | 0.9951 | 0.9992 | 376.9 | +91.2% |
| | endolmspec | 0.0512 | 0.3459 | 4.399 | 0.0713 | 0.9827 | 0.9982 | 0.9995 | 148.2 | +15.3% |
| | iat | 0.0470 | 0.2780 | 3.919 | 0.0645 | 0.8995 | 0.9985 | 0.9997 | 276.1 | +5.9% |

> δ < 1.25ᵏ = fracción de píxeles con max(pred/gt, gt/pred) < 1.25ᵏ (mayor = mejor). ΔAR% negativo = mejora vs. baseline. ← indica mejora.

---

### Hallazgos principales

#### 1. HADepth es el mejor modelo con diferencia
HADepth (DINOv2 + DoRA) alcanza AbsRel=**0.044**, δ<1.25=**0.9945** sin enhancement — mejor que cualquier combinación modelo+enhancement. La segunda mejor opción (AF-SfMLearner+none, 0.106) tiene más del doble de error.

#### 2. Los mejores modelos son perjudicados por el enhancement
AF-SfMLearner y HADepth son los únicos donde el baseline es óptimo. Fueron entrenados con loss functions robustas a especulares: el enhancement modifica la distribución de entrada alejándola del entrenamiento. Retinex es especialmente destructivo (HADepth +91.2%, AF-SfMLearner +19.9%).

#### 3. Los modelos ResNet/genéricos se benefician de IAT
EndoDepth + IAT: **−12.0% AbsRel** (δ<1.25 sube 0.776→0.822). MonoViT + IAT: **−7.3%**. IAT, por su corrección suave (local mul/add + gamma), preserva mejor la distribución de entrada.

#### 4. EndoSfMLearner es completamente inmune
Cambios < 0.1% con cualquier enhancement — su brightness-aware photometric loss lo hace insensible por diseño.

#### 5. EndoLMSPEC nunca es la mejor opción
En todos los modelos es peor que IAT o que no hacer nada (domain gap Endo4IE→SCARED).

---

### Veredicto final

| Modelo | Mejor config. | AbsRel | δ<1.25 | IT (ms) | Conclusión |
|---|---|---|---|---|---|
| **HADepth** | **none** | **0.044** | **0.995** | 32.1 | ★ Mejor absoluto; enhancement perjudica |
| AF-SfMLearner | none | 0.106 | 0.903 | 27.4 | 2º mejor; enhancement perjudica |
| EndoDepth | IAT | 0.145 | 0.822 | 271.4 | Enhancement útil (+IAT) |
| MonoViT | IAT | 0.283 | 0.492 | 297.0 | Solo IAT; otros perjudican |
| EndoSfMLearner | — | 0.291 | 0.471 | 6.2 | Enhancement innecesario |

**La hipótesis se confirma parcialmente con un hallazgo inesperado:** el enhancement mejora modelos genéricos (EndoDepth, MonoViT con IAT), pero los modelos **específicamente diseñados para endoscopía** (HADepth, AF-SfMLearner) no lo necesitan y son perjudicados. La mejor configuración global es HADepth sin ningún enhancement.

> **Avance 5:** este experimento usa los pesos **originales** de cada paper. En el Avance 5 se repite con pesos **re-entrenados en SCARED** (Dr. Espinosa Loera) para medir si el fine-tuning en el dominio de evaluación cambia el comportamiento del enhancement.

## Referencias

- Shao, S., et al. (en revisión). HADepth: Highlight-aware monocular depth estimation for endoscopy. *Signal, Image and Video Processing*.
- Shao, S., et al. (2022). AF-SfMLearner. *Medical Image Analysis*, 77, 102338.
- Zhao, C., et al. (2022). MonoViT. *3DV 2022*. arXiv:2208.03543.
- Ozyoruk, K. B., et al. (2020). EndoSLAM Dataset and Endo-SfMLearner. *arXiv:2006.16670*.
- García-Vega, A., et al. (2022). Multi-Scale Structural-aware Exposure Correction. *arXiv:2210.15033*.
- Wang, T., et al. (2022). Ultra-High-Definition Low-Light Image Enhancement. *AAAI 2022*.
- Recasens, D., et al. (2021). Endo-Depth-and-Motion. *arXiv:2103.16525*.
- Rahman, Z., et al. (2004). Retinex processing. *Journal of Electronic Imaging*, 13(1).
- Allan, M., et al. (2021). SCARED Challenge. *arXiv:2101.01133*.

---
## 12. Exportar resultados al repositorio

Las imágenes ya se guardaron directamente en `outcomes/avance4/` del repo clonado.
Esta celda hace `git add / commit / push`.

**En Colab**: necesita un GitHub token con scope `repo`.
Créalo en https://github.com/settings/tokens y pégalo cuando se pida.

**En local (VS Code)**: el push usa las credenciales de git configuradas, no pide token.

In [ ]:
import subprocess, shutil
from pathlib import Path

def git(args):
    subprocess.check_call(["git", "-C", str(REPO_ROOT)] + args)

# En Colab necesitamos autenticar el push con token
if IN_COLAB:
    from getpass import getpass
    token = getpass("GitHub token (repo scope): ")
    remote_url = f"https://{token}@github.com/jmtoral/proyecto_integrador_52.git"
    git(["remote", "set-url", "origin", remote_url])
    git(["pull", "--rebase"])

# --- Localizar imágenes generadas -----------------------------------------
# Primero busca en OUT_DIR (flujo normal con notebook actualizado)
new_files = list(OUT_DIR.glob("avance4_*.png"))
csv_file  = OUT_DIR / "avance4_results.csv"
if csv_file.exists():
    new_files.append(csv_file)

# Si no hay nada, busca en el path legacy de Drive y copia al repo
if not new_files and IN_COLAB:
    legacy_dir = BASE / "avance4_outputs"
    legacy_files = list(legacy_dir.glob("avance4_*.png"))
    legacy_csv   = legacy_dir / "avance4_results.csv"
    if legacy_csv.exists():
        legacy_files.append(legacy_csv)
    if legacy_files:
        print(f"Encontradas {len(legacy_files)} imagenes en Drive ({legacy_dir})")
        print("Copiando a repo...")
        for f in legacy_files:
            shutil.copy(f, OUT_DIR / f.name)
            print(f"  copiado: {f.name}")
        new_files = list(OUT_DIR.glob("avance4_*.png"))
        csv_file  = OUT_DIR / "avance4_results.csv"
        if csv_file.exists():
            new_files.append(csv_file)

# --------------------------------------------------------------------------
print(f"OUT_DIR: {OUT_DIR}")
print(f"Archivos encontrados: {[f.name for f in new_files] if new_files else 'ninguno'}")

if not new_files:
    print("\nSin imagenes generadas. Corre las celdas de visualizacion primero.")
else:
    git(["add"] + [str(f.relative_to(REPO_ROOT)) for f in new_files])
    changed = subprocess.run(
        ["git", "-C", str(REPO_ROOT), "diff", "--cached", "--name-only"],
        capture_output=True, text=True).stdout.strip()
    if changed:
        git(["commit", "-m", f"Add: resultados Avance4 ({len(new_files)} archivos)"])
        git(["push"])
        print(f"Push OK: {[f.name for f in new_files]}")
    else:
        print("Sin cambios nuevos (archivos ya en el repo)")